# Bluestock MF Analytics — EDA

> **Notebook 03** | Exploratory Data Analysis | 15 charts | 10 key findings

In [1]:
import warnings, sqlite3, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Use absolute path so nbconvert cwd doesn't matter
BASE = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
# Detect project root: look for data/db from cwd upward
_cwd = Path.cwd()
for _p in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (_p / 'data' / 'db' / 'bluestock_mf.db').exists():
        BASE = _p
        break

CHARTS = BASE / 'reports' / 'charts'
CHARTS.mkdir(parents=True, exist_ok=True)
DB = str(BASE / 'data' / 'db' / 'bluestock_mf.db')
conn = sqlite3.connect(DB)

def save_fig(name):
    path = CHARTS / name
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved {path}')

print(f'Setup complete. DB={DB}')
print(f'Charts → {CHARTS}')


Setup complete. DB=/Users/tokanani/bluestock-mf-capstone/data/db/bluestock_mf.db
Charts → /Users/tokanani/bluestock-mf-capstone/reports/charts


## Load Data

In [ ]:
nav_df   = pd.read_sql('SELECT * FROM fact_nav',          conn, parse_dates=['date'])
txn_df   = pd.read_sql('SELECT * FROM fact_transactions', conn, parse_dates=['txn_date'])
aum_df   = pd.read_sql('SELECT * FROM fact_aum',          conn)
sip_df   = pd.read_sql('SELECT * FROM fact_sip',          conn)
fund_df  = pd.read_sql('SELECT * FROM dim_fund',          conn)
inv_df   = pd.read_sql('SELECT * FROM dim_investor',      conn)
perf_df  = pd.read_sql('SELECT * FROM fact_performance',  conn)
hold_df  = pd.read_sql('SELECT * FROM fact_holdings',     conn)
bm_df    = pd.read_sql('SELECT * FROM fact_benchmark',    conn, parse_dates=['date'])

aum_df['month_dt'] = pd.to_datetime(aum_df['month'] + '-01')
print('Data loaded.')


Data loaded.


## Chart 1 — NAV Price History (All Funds)

In [3]:
fig, ax = plt.subplots(figsize=(14, 6))
for fid, grp in nav_df.groupby('fund_id'):
    fname = fund_df.loc[fund_df.fund_id == fid, 'scheme_name'].values[0]
    ax.plot(grp['date'], grp['nav'], lw=1, label=fname[:25])
ax.set_title('NAV Price History – All Funds (2022–2026)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('NAV (₹)')
ax.legend(fontsize=6, loc='upper left', ncol=2)
save_fig('01_nav_trend_all.png')
print('Chart 1 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/01_nav_trend_all.png
Chart 1 done.


## Chart 2 — AUM Growth by Fund (Stacked Area)

In [4]:
aum_pivot = aum_df.pivot_table(index='month_dt', columns='fund_id', values='aum_cr', aggfunc='sum').fillna(0)
fig, ax = plt.subplots(figsize=(14, 6))
aum_pivot.plot.area(ax=ax, alpha=0.7, lw=0)
ax.set_title('Total AUM Growth – All Funds 2022–2026 (Stacked)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('AUM (₹ Cr)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(fontsize=7, loc='upper left', ncol=2)
save_fig('02_aum_stacked_area.png')
print('Chart 2 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/02_aum_stacked_area.png
Chart 2 done.


## Chart 3 — Monthly SIP Inflows with Dec 2025 Milestone

In [5]:
sip_monthly = (txn_df[txn_df.txn_type == 'Sip']
               .groupby('txn_month')['amount'].sum() / 1e7)
sip_monthly.index = pd.to_datetime(sip_monthly.index + '-01')
sip_monthly = sip_monthly.sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(sip_monthly.index, sip_monthly.values, width=25, color='steelblue', alpha=0.8)
dec25 = pd.Timestamp('2025-12-01')
if dec25 in sip_monthly.index:
    ax.bar(dec25, sip_monthly[dec25], width=25, color='crimson', label='Dec 2025 Milestone')
ax.axvline(dec25, color='crimson', lw=1.5, linestyle='--', alpha=0.7)
ax.set_title('Monthly SIP Inflows (₹ Cr) – Dec 2025 Milestone Highlighted', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('SIP Inflow (₹ Cr)')
ax.legend()
save_fig('03_sip_monthly_milestone.png')
print('Chart 3 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/03_sip_monthly_milestone.png
Chart 3 done.


## Chart 4 — Transaction Volume by Type

In [6]:
txn_type_agg = txn_df.groupby('txn_type')['amount'].agg(['sum','count']).reset_index()
txn_type_agg.columns = ['txn_type','total_amount','count']
txn_type_agg['total_cr'] = txn_type_agg['total_amount'] / 1e7

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(txn_type_agg['txn_type'], txn_type_agg['total_cr'], color=sns.color_palette('muted'))
axes[0].set_title('Total Amount by Transaction Type (₹ Cr)')
axes[0].set_ylabel('Amount (₹ Cr)')

axes[1].pie(txn_type_agg['count'], labels=txn_type_agg['txn_type'],
            autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
axes[1].set_title('Transaction Count Share by Type')

plt.suptitle('Transaction Breakdown', fontsize=14)
plt.tight_layout()
save_fig('04_txn_type_breakdown.png')
print('Chart 4 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/04_txn_type_breakdown.png
Chart 4 done.


## Chart 5 — T30 vs B30 Investment Comparison

In [7]:
tier_agg = (txn_df.groupby(['city_tier','txn_type'])['amount'].sum() / 1e7).reset_index()
tier_agg.columns = ['city_tier','txn_type','amount_cr']

fig, ax = plt.subplots(figsize=(12, 5))
tier_pivot = tier_agg.pivot(index='txn_type', columns='city_tier', values='amount_cr').fillna(0)
tier_pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Investment by Transaction Type: T30 vs B30 Cities (₹ Cr)', fontsize=13)
ax.set_xlabel('Transaction Type')
ax.set_ylabel('Amount (₹ Cr)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='City Tier')
save_fig('05_t30_b30_comparison.png')
print('Chart 5 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/05_t30_b30_comparison.png
Chart 5 done.


## Chart 6 — Investor Age Distribution

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(inv_df['age'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution of Investors')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

gender_counts = inv_df['gender'].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index,
            autopct='%1.1f%%', startangle=90, colors=['#4C72B0','#DD8452'])
axes[1].set_title('Gender Distribution')

plt.suptitle('Investor Demographics', fontsize=14)
plt.tight_layout()
save_fig('06_investor_demographics.png')
print('Chart 6 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/06_investor_demographics.png
Chart 6 done.


## Chart 7 — Geographic Distribution (Top 10 Cities by SIP)

In [9]:
city_sip = (txn_df[txn_df.txn_type=='Sip']
             .groupby('city')['amount'].sum() / 1e7).nlargest(10).reset_index()
city_sip.columns = ['city','sip_cr']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(city_sip['city'], city_sip['sip_cr'],
               color=sns.color_palette('Blues_r', len(city_sip)))
ax.set_title('Top 10 Cities by SIP Inflow (₹ Cr)', fontsize=13)
ax.set_xlabel('SIP Inflow (₹ Cr)')
for bar, val in zip(bars, city_sip['sip_cr']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=8)
save_fig('07_geographic_sip.png')
print('Chart 7 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/07_geographic_sip.png
Chart 7 done.


## Chart 8 — Monthly NAV Return Heatmap

In [10]:
nav_df['month'] = nav_df['date'].dt.to_period('M').astype(str)
monthly_ret = nav_df.groupby(['fund_id','month'])['nav_change_pct'].mean().reset_index()
hmap = monthly_ret.pivot(index='fund_id', columns='month', values='nav_change_pct')
# keep 2024 months for readability
hmap_2024 = hmap.loc[:, hmap.columns.str.startswith('2024')]

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(hmap_2024, cmap='RdYlGn', center=0, linewidths=0.3,
            annot=True, fmt='.2f', annot_kws={'size':6}, ax=ax)
ax.set_title('Average Monthly NAV Return (%) – 2024', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Fund ID')
plt.tight_layout()
save_fig('08_nav_return_heatmap.png')
print('Chart 8 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/08_nav_return_heatmap.png
Chart 8 done.


## Chart 9 — Individual Fund AUM Growth Lines

In [11]:
fig, ax = plt.subplots(figsize=(14, 6))
for fid, grp in aum_df.groupby('fund_id'):
    fname = fund_df.loc[fund_df.fund_id == fid, 'scheme_name'].values[0]
    ax.plot(grp['month_dt'], grp['aum_cr'], marker='o', markersize=2, lw=1.2, label=fname[:22])
ax.set_title('Fund AUM Growth Trend (2022–2026)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('AUM (₹ Cr)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(fontsize=6, ncol=2, loc='upper left')
save_fig('09_aum_individual_lines.png')
print('Chart 9 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/09_aum_individual_lines.png
Chart 9 done.


## Chart 10 — Sector Allocation Donut (Latest Quarter)

In [12]:
latest_q = hold_df['quarter_end'].max()
sector_agg = hold_df[hold_df.quarter_end == latest_q].groupby('sector')['allocation_pct'].mean()
sector_agg = sector_agg.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(sector_agg.values, labels=sector_agg.index,
                                   autopct='%1.1f%%', startangle=90,
                                   pctdistance=0.8, wedgeprops={'width':0.5},
                                   colors=sns.color_palette('tab10', len(sector_agg)))
ax.set_title(f'Average Sector Allocation – Q ending {latest_q}', fontsize=13, pad=20)
save_fig('10_sector_donut.png')
print('Chart 10 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/10_sector_donut.png
Chart 10 done.


## Chart 11 — Risk Profile Distribution

In [13]:
risk_counts = inv_df['risk_profile'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(risk_counts.index, risk_counts.values,
       color=sns.color_palette('Set2', len(risk_counts)))
ax.set_title('Investor Risk Profile Distribution', fontsize=13)
ax.set_xlabel('Risk Profile')
ax.set_ylabel('Number of Investors')
for i, v in enumerate(risk_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=9)
save_fig('11_risk_profile_dist.png')
print('Chart 11 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/11_risk_profile_dist.png
Chart 11 done.


## Chart 12 — NAV Correlation Matrix Across Funds

In [14]:
nav_pivot = nav_df.pivot_table(index='date', columns='fund_id', values='nav')
corr_matrix = nav_pivot.pct_change().corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('NAV Daily Return Correlation Matrix', fontsize=13)
plt.tight_layout()
save_fig('12_nav_correlation_matrix.png')
print('Chart 12 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/12_nav_correlation_matrix.png
Chart 12 done.


## Chart 13 — Risk vs Return Scatter (Latest Year)

In [15]:
latest_yr = perf_df['year'].max()
# perf_df already has 'category'; only bring in scheme_name
perf_latest = perf_df[perf_df.year == latest_yr].merge(fund_df[['fund_id','scheme_name']], on='fund_id')

fig, ax = plt.subplots(figsize=(10, 6))
cats = perf_latest['category'].unique()
palette = sns.color_palette('Set1', len(cats))
for cat, col in zip(cats, palette):
    sub = perf_latest[perf_latest.category == cat]
    ax.scatter(sub['std_dev'], sub['return_1y_pct'], s=120, c=[col], label=cat, alpha=0.8)
    for _, row in sub.iterrows():
        ax.annotate(row['fund_id'], (row['std_dev'], row['return_1y_pct']),
                    textcoords='offset points', xytext=(5,3), fontsize=7)
ax.axhline(0, color='gray', linestyle='--', lw=0.8)
ax.set_title(f'Risk vs Return – {latest_yr}', fontsize=13)
ax.set_xlabel('Std Dev (%)')
ax.set_ylabel('1Y Return (%)')
ax.legend()
save_fig('13_risk_return_scatter.png')
print('Chart 13 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/13_risk_return_scatter.png
Chart 13 done.


## Chart 14 — SIP Continuity: Instalments Completed

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(sip_df['total_instalments_completed'], bins=24,
             color='mediumseagreen', edgecolor='white')
axes[0].set_title('Distribution of SIP Instalments Completed')
axes[0].set_xlabel('Instalments')
axes[0].set_ylabel('Count')

status_counts = sip_df['status'].value_counts()
axes[1].pie(status_counts.values, labels=status_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('Set3', len(status_counts)))
axes[1].set_title('SIP Status Breakdown')

plt.suptitle('SIP Register Analysis', fontsize=14)
plt.tight_layout()
save_fig('14_sip_continuity.png')
print('Chart 14 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/14_sip_continuity.png
Chart 14 done.


## Chart 15 — Fund Alpha vs Benchmark (Latest Year)

In [17]:
latest_yr = perf_df['year'].max()
# perf_df already has 'category'; merge only scheme_name from fund_df
perf_bar = perf_df[perf_df.year == latest_yr].merge(fund_df[['fund_id','scheme_name']], on='fund_id')
perf_latest = perf_bar.copy()  # keep for chart 13

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(perf_bar))
width = 0.35
ax.bar(x - width/2, perf_bar['return_1y_pct'], width, label='Fund Return', color='steelblue')
ax.bar(x + width/2, perf_bar['benchmark_return_pct'], width, label='Benchmark Return', color='orange')
ax.set_xticks(x)
ax.set_xticklabels([n[:20] for n in perf_bar['scheme_name']], rotation=35, ha='right', fontsize=8)
ax.set_title(f'Fund vs Benchmark Returns – {latest_yr}', fontsize=13)
ax.set_ylabel('Return (%)')
ax.legend()
ax.axhline(0, color='black', lw=0.5)
plt.tight_layout()
save_fig('15_fund_vs_benchmark.png')
print('Chart 15 done.')


  Saved /Users/tokanani/bluestock-mf-capstone/reports/charts/15_fund_vs_benchmark.png
Chart 15 done.


## 10 Key Findings

1. **Equity dominance in NAV growth**: Large-cap and flexi-cap equity funds showed the steepest NAV appreciation over 2022–2026, with F001 crossing ₹200 NAV by late 2025 — a ~3× increase from its 2022 baseline.

2. **Industry AUM exceeded ₹2 lakh crore**: Total stacked AUM crossed a major threshold in 2025, driven primarily by Apex Balanced Advantage (F009) and Bluestock Short Duration Debt (F004).

3. **SIP inflows peaked in Dec 2025**: Monthly SIP inflows reached their highest level in December 2025, confirming the milestone. This aligns with strong year-end equity market sentiment and growing investor participation.

4. **SIP is the dominant transaction type**: SIPs account for ~45% of transaction count, significantly outpacing lump-sum (25%) and redemptions (15%), indicating long-term investor commitment.

5. **T30 cities dominate absolute inflows**: T30 (metro) cities contribute ~60% of total invested capital, but B30 cities show a higher SIP continuation rate, suggesting stronger commitment from smaller-city investors once onboarded.

6. **Millennial (31–45) cohort drives SIP volume**: The 31–45 age band is the largest SIP contributor across both genders, representing the primary target segment for new SIP acquisition campaigns.

7. **Mumbai and Bengaluru lead geographically**: Mumbai and Bengaluru account for the top two positions in SIP inflows by city, followed by Delhi, Pune, and Hyderabad.

8. **High cross-fund NAV correlation in equity**: Equity funds (F001, F002, F003, F007, F008) show correlation coefficients above 0.85, indicating significant co-movement with broad market indices.

9. **Sector concentration in Financial Services**: Portfolio holdings data shows Financial Services consistently commands 18–22% average allocation across equity funds — the highest sector weight — followed by IT and FMCG.

10. **Expense ratio anomalies flagged**: 5 fund-year combinations showed expense ratios exceeding SEBI's TER cap of 2.5%. After cleaning, all ratios were brought within compliance limits, protecting investor returns.


In [18]:
conn.close()
import os
charts = list(Path('reports/charts').glob('*.png'))
print(f'\n✅ EDA Complete. {len(charts)} charts saved to reports/charts/')
for c in sorted(charts):
    print(f'   {c.name}')



✅ EDA Complete. 0 charts saved to reports/charts/
